# Collect metrics from Drive (no GPU needed)

gsplat writes `stats/train_step*.json` (train time, peak VRAM) and `stats/val_step*.json` (scores, #Gaussians) for every run. Some README rows are missing train time and VRAM because older notebooks didn't print them, but the files are still on Drive. This notebook reads them all.

**Run:** `Runtime → Change runtime type → CPU`, then `Run all`. Paste the table back into the chat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, re
RESULTS = '/content/drive/MyDrive/3dgs-digital-twin/results'
step_of = lambda p: int(re.search(r'step(\d+)', os.path.basename(p)).group(1))
print('| Run | Steps | #Gaussians | PSNR | SSIM | LPIPS | Train | Peak VRAM | Render FPS |')
print('|---|---|---|---|---|---|---|---|---|')
for rd in sorted(glob.glob(f'{RESULTS}/*/')):
    for v in sorted(glob.glob(f'{rd}stats/val_step*.json'), key=step_of):
        s = step_of(v); val = json.load(open(v))
        tr = glob.glob(f'{rd}stats/train_step{s:04d}_rank*.json')
        t = json.load(open(tr[0])) if tr else {}
        lp = f"{val['lpips']:.3f}" if 'lpips' in val else '–'
        print(f"| {os.path.basename(rd.rstrip('/'))} | {s + 1:,} | {val.get('num_GS', 0):,} | {val['psnr']:.2f} | "
              f"{val['ssim']:.3f} | {lp} | {t.get('ellipse_time', float('nan')) / 60:.1f} min | "
              f"{t.get('mem', float('nan')):.2f} GB | {1 / val['ellipse_time']:.0f} |" if 'ellipse_time' in val else
              f"| {os.path.basename(rd.rstrip('/'))} | {s + 1:,} | {val.get('num_GS', 0):,} | {val['psnr']:.2f} | "
              f"{val['ssim']:.3f} | {lp} | {t.get('ellipse_time', float('nan')) / 60:.1f} min | {t.get('mem', float('nan')):.2f} GB | – |")